# 05 — Prompt Comparison & Freezing

This notebook aggregates the prompt-development results from all 4 models and selects the best prompt per level per model.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "jsonlines", "jiwer"])

print("✅ Dependencies ready.")


In [ ]:
import pandas as pd
import os, json

RESULTS_BASE = "/kaggle/working/results/prompt_dev"
MODELS = ["gemma4_12b", "qwen2_5_omni", "voxtral_mini", "phi4_multimodal"]

all_summaries = []
for model in MODELS:
    path = f"{RESULTS_BASE}/{model}/prompt_summary.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["model"] = model
        all_summaries.append(df)
        print(f"✅ Loaded {model}: {len(df)} prompt variants")
    else:
        print(f"⚠️  Missing: {path}")

if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)
    print(f"\n📊 Combined results: {len(combined)} rows")
    display(combined.sort_values(["model", "prompt_id"]))
else:
    print("No results found. Run notebooks 01-04 first.")
    combined = pd.DataFrame()


## Select Best Prompt Per Level Per Model


In [ ]:
if len(combined) > 0:
    # Extract level from prompt_id
    combined["level"] = combined["prompt_id"].str.split("_").str[0]
    
    # For each model+level, pick the prompt with lowest avg_wer
    best = combined.sort_values("avg_wer").groupby(["model", "level"]).first().reset_index()
    print("\n🏆 Best prompt per level per model:")
    display(best[["model", "level", "prompt_id", "avg_wer", "avg_cer", "count"]])
    
    # Build frozen registry
    registry = {"models": {}}
    for _, row in best.iterrows():
        if row["model"] not in registry["models"]:
            registry["models"][row["model"]] = {}
        registry["models"][row["model"]][row["level"]] = row["prompt_id"]
    
    os.makedirs("/kaggle/working/configs", exist_ok=True)
    with open("/kaggle/working/configs/frozen_prompt_registry.json", "w") as f:
        json.dump(registry, f, indent=2)
    
    print("\n📁 Frozen registry saved to configs/frozen_prompt_registry.json")
    print(json.dumps(registry, indent=2))
else:
    print("No data to process.")


## Status & Failure Summary


In [ ]:
for model in MODELS:
    path = f"{RESULTS_BASE}/{model}/utterance_metrics.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\n=== {model} ===")
        print(df["status"].value_counts())
    else:
        print(f"{model}: no data")
